# 🎨 PhotoEnhance — AI Photo Upscaler
**Безкоштовне покращення якості фото за допомогою Real-ESRGAN**

---

| Параметр | Значення |
|---|---|
| Модель | Real-ESRGAN |
| GPU | Google Colab T4 (16GB) |
| Масштаб | x2, x4, x8 |
| Формати | JPG, PNG, WEBP |
| Вартість | $0 |

> ⚠️ **Переконайся, що увімкнено GPU:** Runtime → Change runtime type → T4 GPU

## Як користуватись: 3 кроки
| Крок | Що робить |
|---|---|
| ▶ **Крок 1** | Встановлення, KeepAlive, перевірка GPU |
| ▶ **Крок 2** | Google Drive + завантаження фото |
| ▶ **Крок 3** | Обробка та скачування результату |


In [ ]:
# @title ⚙️ Крок 1 — Встановлення та перевірка { display-mode: "form" }
# @markdown Запускай **один раз** на початку кожної Colab-сесії.
# @markdown Повторний запуск безпечний — все вже встановлене буде пропущено.

# ══════════════════════════════════════════════════════════════
# STUDENT GPU TIPS (безкоштовно або майже безкоштовно):
#
#  ☁️  Google Colab       — T4 безкоштовно | A100 через Colab Pro
#  🏆  Kaggle Notebooks   — T4/P100, 30 год/тиждень, безкоштовно
#      (File → Settings → Accelerator → GPU T4x2)
#  ⚡  Lightning.ai       — A10G, безкоштовний tier
#  🎓  GitHub Education   — Student Pack → AWS/Azure/GCP кредити
#      (github.com/education, підтвердь email університету)
#  🚀  Vast.ai            — оренда GPUs ~$0.2/год (Colab Pro дешевше)
# ══════════════════════════════════════════════════════════════

import sys
import os
import subprocess
import threading

import ipywidgets as widgets
from IPython.display import display, Javascript, HTML

# ─── прогрес-бар ────────────────────────────────────────────────────
_pb = widgets.IntProgress(
    value=0, min=0, max=100,
    description='⏳  0%', bar_style='info',
    style={'bar_color': '#4a90e2', 'description_width': '65px'},
    layout=widgets.Layout(width='100%', height='28px'),
)
_lbl = widgets.HTML('<span style="font-size:12px;color:#555;">Перевірка середовища...</span>')
display(widgets.VBox([_pb, _lbl]))

# ════════════════════════════════════════════════
# §0  ПЕРЕВІРКИ СЕРЕДОВИЩА
# ════════════════════════════════════════════════
def _check(cond, ok_msg, fail_msg, fatal=False):
    if cond:
        print(f'✅ {ok_msg}')
    else:
        print(f'{"❌" if fatal else "⚠️"} {fail_msg}')
    return cond

_ready = True

# Python версія
_v = sys.version_info
_ready &= _check(
    _v >= (3, 8),
    f'Python {_v.major}.{_v.minor}.{_v.micro}',
    f'Python {_v.major}.{_v.minor} — потрібен 3.8+.',
    fatal=True,
)

# Colab (не зберігаємо модуль — лише перевірка)
try:
    __import__('google.colab')  # Colab env check (side-effect import)
    print('✅ Середовище: Google Colab')
except ImportError:
    print('⚠️  Не Colab — деякі функції (Drive, download) не працюватимуть.')

# RAM
try:
    with open('/proc/meminfo') as _mf:
        _ram_gb = int([l for l in _mf if 'MemAvailable' in l][0].split()[1]) / 1048576
    _check(_ram_gb >= 4,
           f'RAM вільно: {_ram_gb:.1f} GB',
           f'Мало RAM: {_ram_gb:.1f} GB — можливі OOM при ×8.')
except Exception:
    print('ℹ️  RAM: не вдалося перевірити')

# Диск
try:
    import shutil as _sh
    _, _, _fd = _sh.disk_usage('/content')
    _check(_fd / 1073741824 >= 3,
           f'Диск вільно: {_fd / 1073741824:.1f} GB',
           f'Мало місця: {_fd / 1073741824:.1f} GB — можливі помилки запису.')
except Exception:
    print('ℹ️  Диск: не вдалося перевірити')

if not _ready:
    _pb.bar_style = 'danger'
    raise SystemExit('❌ Усунь помилки вище та перезапусти Крок 1.')

_pb.value = 5; _pb.description = '🔎  5%'
_lbl.value = '<span style="font-size:12px;color:#555;">Встановлення залежностей...</span>'
print('\n📦 Починаємо встановлення...\n')

# ════════════════════════════════════════════════
# §1  ЗАЛЕЖНОСТІ
# ════════════════════════════════════════════════
os.environ['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'
os.environ['PIP_NO_WARN_SCRIPT_LOCATION']   = '1'

print('📦 basicsr==1.4.2 / facexlib / gfpgan...')
get_ipython().run_line_magic('pip', 'install -q "basicsr==1.4.2" facexlib gfpgan')  # type: ignore
_pb.value = 18; _pb.description = '⬇️  18%'

print('📦 Pillow / opencv / ipywidgets...')
get_ipython().run_line_magic('pip', 'install -q "Pillow>=9" "opencv-python-headless>=4.8" ipywidgets')  # type: ignore
_pb.value = 23; _pb.description = '⬇️  23%'

# ── torchvision >= 0.16 compat fix (basicsr шукає старий module) ───
from types import ModuleType
import torchvision.transforms.functional as _tvf
_compat = ModuleType('torchvision.transforms.functional_tensor')
_compat.rgb_to_grayscale = _tvf.rgb_to_grayscale  # type: ignore
sys.modules.setdefault('torchvision.transforms.functional_tensor', _compat)
print('✅ torchvision compat patch OK')
_pb.value = 26; _pb.description = '📦  26%'

# ════════════════════════════════════════════════
# §2  Real-ESRGAN
# ════════════════════════════════════════════════
_lbl.value = '<span style="font-size:12px;color:#555;">Клонування Real-ESRGAN...</span>'
if not os.path.isdir('/content/Real-ESRGAN'):
    print('📦 Клонування Real-ESRGAN...')
    get_ipython().system('git clone -q https://github.com/xinntao/Real-ESRGAN.git /content/Real-ESRGAN')  # type: ignore
    _pb.value = 33; _pb.description = '📦  33%'
    get_ipython().run_line_magic('pip', 'install -q -r /content/Real-ESRGAN/requirements.txt')  # type: ignore
else:
    print('✅ Real-ESRGAN вже є')
os.chdir('/content/Real-ESRGAN')
# pip install -e є надійнішим за setup.py develop і безпечний для повторного запуску
print('📦 pip install -e Real-ESRGAN...')
get_ipython().run_line_magic('pip', 'install -q -e /content/Real-ESRGAN')  # type: ignore
print('✅ Real-ESRGAN встановлено')
_pb.value = 45; _pb.description = '📦  45%'

# ════════════════════════════════════════════════
# §3  CodeFormer
# ════════════════════════════════════════════════
_lbl.value = '<span style="font-size:12px;color:#555;">Клонування CodeFormer...</span>'
if not os.path.isdir('/content/CodeFormer'):
    print('🔬 Клонування CodeFormer...')
    get_ipython().system('git clone -q https://github.com/sczhou/CodeFormer.git /content/CodeFormer')  # type: ignore
    _pb.value = 52; _pb.description = '🔬  52%'
    os.chdir('/content/CodeFormer')
    get_ipython().run_line_magic('pip', 'install -q -r requirements.txt')  # type: ignore
    get_ipython().system('python basicsr/setup.py develop -q 2>/dev/null')  # type: ignore
    _pb.value = 60; _pb.description = '🔬  60%'
    _lbl.value = '<span style="font-size:12px;color:#555;">Ваги facelib та CodeFormer...</span>'
    get_ipython().system('python scripts/download_pretrained_models.py facelib')  # type: ignore
    _pb.value = 72; _pb.description = '⬇️  72%'
    get_ipython().system('python scripts/download_pretrained_models.py CodeFormer')  # type: ignore
    print('✅ CodeFormer встановлено')
else:
    print('✅ CodeFormer вже є')
_pb.value = 82; _pb.description = '🔬  82%'

# Re-pin basicsr==1.4.2 — requirements.txt може встановити новішу версію (без data.degradations)
print('📦 Re-pin basicsr==1.4.2 (фіналізація)...')
get_ipython().run_line_magic('pip', 'install -q "basicsr==1.4.2"')  # type: ignore
print('✅ basicsr==1.4.2 OK')

# ── Патч: CodeFormer має власний (неповний) basicsr ─────────────────
# /content/CodeFormer стоїть першим у sys.path → Python знаходить
# CodeFormer basicsr замість повного pip basicsr==1.4.2.
#
# ПРАВИЛЬНЕ РІШЕННЯ:
#   1. Копіюємо CodeFormer-специфічні файли (codeformer_arch тощо)
#      у ПОВНИЙ pip basicsr.
#   2. Перейменовуємо /content/CodeFormer/basicsr → basicsr_cf_bak,
#      щоб Python більше не знаходив неповну версію першою.
import sysconfig as _sc
import shutil as _sh2
import glob as _gl

_site    = _sc.get_paths()['purelib']
_pip_bsr = os.path.join(_site, 'basicsr')
if not os.path.isdir(_pip_bsr):
    _found = _gl.glob('/usr/local/lib/python*/dist-packages/basicsr')
    _pip_bsr = _found[0] if _found else ''

_cf_bsr     = '/content/CodeFormer/basicsr'
_cf_bsr_bak = '/content/CodeFormer/basicsr_cf_bak'
_patched    = 0

# Файли, де CF-версія є надмножиною pip-версії (містить додаткові функції)
# → завжди перезаписуємо pip-версію CF-версією
_CF_OVERWRITE = {
    os.path.join('data', 'data_util.py'),
    os.path.join('utils', 'misc.py'),
}

if _pip_bsr and os.path.isdir(_pip_bsr) and os.path.isdir(_cf_bsr):
    # Крок 1: копіюємо CF-файли → pip basicsr
    #   - нові файли: завжди копіюємо
    #   - файли з _CF_OVERWRITE: перезаписуємо (CF є надмножиною pip)
    for _root, _dirs, _files in os.walk(_cf_bsr):
        _rel     = os.path.relpath(_root, _cf_bsr)
        _dst_dir = os.path.join(_pip_bsr, _rel)
        os.makedirs(_dst_dir, exist_ok=True)
        for _f in _files:
            if _f.endswith('.pyc'):
                continue
            _rel_f = os.path.join(_rel, _f) if _rel != '.' else _f
            _dst_f = os.path.join(_dst_dir, _f)
            _force = _rel_f in _CF_OVERWRITE
            if not os.path.isfile(_dst_f) or _force:
                _sh2.copy2(os.path.join(_root, _f), _dst_f)
                _patched += 1
    # Крок 2: ховаємо CF basicsr щоб Python не знаходив його першим
    if os.path.isdir(_cf_bsr_bak):
        _sh2.rmtree(_cf_bsr_bak)
    os.rename(_cf_bsr, _cf_bsr_bak)
    print(f'✅ Patch OK: {_patched} CF-файлів → pip basicsr; CF basicsr схований')
elif not os.path.isdir(_cf_bsr):
    # Already hidden — re-apply overwrite files from backup so they are always up to date
    if _pip_bsr and os.path.isdir(_cf_bsr_bak):
        for _rel_f in _CF_OVERWRITE:
            _src_f = os.path.join(_cf_bsr_bak, _rel_f)
            _dst_f = os.path.join(_pip_bsr, _rel_f)
            if os.path.isfile(_src_f):
                _sh2.copy2(_src_f, _dst_f)
                _patched += 1
    if _patched:
        print(f'✅ CF basicsr схований; {_patched} файлів оновлено з бекапу')
    else:
        print('✅ CF basicsr вже схований (нічого не змінилось)')
else:
    print('⚠️  pip basicsr не знайдено — пропускаємо патч')

# Обов'язково: очищаємо sys.modules від basicsr — Python міг
# закешувати CF-версію ще під час pip install. Без цього rename
# не допомагає: модуль залишається старим у пам'яті.
_purged = [k for k in list(sys.modules) if k == 'basicsr' or k.startswith('basicsr.')]
for k in _purged:
    del sys.modules[k]
if _purged:
    print(f'🔄 Purged {len(_purged)} basicsr-модулів з sys.modules → свіжий import з pip')

os.chdir('/content/Real-ESRGAN')
for _p in ['/content/Real-ESRGAN', '/content/CodeFormer']:
    if _p not in sys.path:
        sys.path.insert(0, _p)

# ════════════════════════════════════════════════
# §4  KeepAlive
# ════════════════════════════════════════════════
_pb.value = 86; _pb.description = '🔌  86%'
display(Javascript("""
(function() {
    if (window._pe_ka) clearInterval(window._pe_ka);
    window._pe_ka = setInterval(function() {
        fetch('/api/kernels').catch(function(){});
    }, 45000);
    console.log('[PhotoEnhance] KeepAlive started (45s)');
})();
"""))

_ka_stop = threading.Event()
def _ka_heartbeat():
    while not _ka_stop.wait(timeout=50):
        pass  # python-side ping щоб kernel не відключився

threading.Thread(target=_ka_heartbeat, daemon=True, name='PE-KeepAlive').start()
print('🔌 KeepAlive активовано')

# ════════════════════════════════════════════════
# §5  GPU + PyTorch
# ════════════════════════════════════════════════
_pb.value = 90; _pb.description = '🖥️  90%'
_lbl.value = '<span style="font-size:12px;color:#555;">Перевірка GPU...</span>'
import torch

torch.backends.cudnn.benchmark        = True
torch.backends.cudnn.allow_tf32       = True
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision('high')
    print('✅ matmul precision=high (TF32)')
except AttributeError:
    print('ℹ️  matmul TF32 via cudnn (PyTorch < 2.0)')

print('=' * 52)
print('🖥️  GPU ІНФОРМАЦІЯ')
print('=' * 52)

if torch.cuda.is_available():
    _gname = torch.cuda.get_device_name(0)
    _gmem  = torch.cuda.get_device_properties(0).total_memory / 1073741824
    _cc    = torch.cuda.get_device_capability(0)

    _nsmi = subprocess.run(
        ['nvidia-smi', '--query-gpu=memory.free,temperature.gpu',
         '--format=csv,noheader,nounits'],
        capture_output=True, text=True,
    )
    _vfree, _temp = '?', '?'
    if _nsmi.returncode == 0:
        _pts = _nsmi.stdout.strip().split(', ')
        if len(_pts) >= 2:
            try:
                _vfree = f'{int(_pts[0]) / 1024:.1f}'
                _temp  = _pts[1]
            except (ValueError, IndexError):
                pass

    print(f'✅ GPU:  {_gname}')
    print(f'💾 VRAM: {_gmem:.1f} GB  |  вільно: {_vfree} GB')
    print(f'🌡️  Temp: {_temp}°C  |  Compute: {_cc[0]}.{_cc[1]}')
    print(f'🔧 CUDA: {torch.version.cuda}  |  PyTorch: {torch.__version__}')
    if _cc[0] >= 8:
        print('🚀 TF32 активний (Ampere/Hopper) — +20% при fp32')
    else:
        print(f'ℹ️  Compute {_cc[0]}.{_cc[1]} — FP16 дає 4–8× прискорення')
    if _gmem < 4:
        print('⚠️  VRAM < 4 GB — при ×8 можливий OOM. Використовуй ×2/×4.')
    print('\n✅ Готово!  Переходь до Кроку 2 ▶')
else:
    print('❌ GPU не знайдено!')
    print('👉 Runtime → Change runtime type → T4 GPU → Save → перезапусти Крок 1')
    print('💡 Kaggle: Settings → Accelerator → GPU T4×2 (безкоштовно 30 год/тиж)')

print('=' * 52)
_pb.value = 100; _pb.description = '✅ 100%'; _pb.bar_style = 'success'
_lbl.value = '<span style="font-size:12px;color:#2ecc71;font-weight:bold;">✅ Крок 1 завершено!</span>'

_html_step1 = (
    '<div style="margin-top:14px;padding:14px 20px;'
    'background:linear-gradient(135deg,#1a1a2e,#16213e);'
    'border-radius:10px;border-left:5px solid #2ecc71;font-family:sans-serif;">'
    '<div style="color:#2ecc71;font-size:18px;font-weight:bold;">✅ Крок 1 завершено!</div>'
    '<div style="color:#ccc;margin-top:6px;font-size:13px;">'
    '📦 Real-ESRGAN ✔&nbsp;&nbsp; 🔬 CodeFormer ✔&nbsp;&nbsp; 🔌 KeepAlive ✔&nbsp;&nbsp;'
    '🖥️ GPU ✔&nbsp;&nbsp; ⚡ TF32/FP16 ✔'
    '</div>'
    '<div style="margin-top:10px;font-size:15px;color:#4a90e2;font-weight:bold;">'
    '↓ &nbsp;Гортай вниз і запускай <span style="color:#f9ca24;">Крок 2</span>'
    '</div></div>'
)
display(HTML(_html_step1))


In [ ]:
# @title 🖼️ Крок 2 — Google Drive + Завантаження фото { display-mode: "form" }
# @markdown Drive необов'язковий — без нього файл скачається напряму в Кроці 3.

import sys
import os
import io
import shutil

import ipywidgets as widgets
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, HTML, clear_output

# ════════════════════════════════════════════════
# §0  Перевірка Кроку 1
# ════════════════════════════════════════════════
if 'torch' not in sys.modules:
    raise SystemExit(
        '❌ Крок 1 не виконано! Запусти Крок 1 і дочекайся зеленої галочки.'
    )

import torch as _t
if _t.cuda.is_available():
    _vram = _t.cuda.get_device_properties(0).total_memory / 1073741824
    print(f'✅ GPU: {_t.cuda.get_device_name(0)}  ({_vram:.1f} GB VRAM)')
    if _vram < 4:
        print('⚠️  VRAM < 4 GB — при ×8 може бути OOM. Використовуй ×2 або ×4.')
else:
    print('⚠️  GPU не знайдено — обробка буде дуже повільною.')
    print('   Runtime → Change runtime type → T4 GPU → Save → перезапусти крок 1.')

# Переконуємось що шлях є до запуску перевірки
for _p in ('/content/Real-ESRGAN', '/content/CodeFormer'):
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)

# Страхувальне очищення: якщо basicsr закешований з CF-версії —
# видаляємо, щоб наступний import пішов у pip-версію
_bsr_purge = [k for k in list(sys.modules) if k == 'basicsr' or k.startswith('basicsr.')]
for k in _bsr_purge:
    del sys.modules[k]

for _lib in ('realesrgan', 'basicsr', 'cv2', 'PIL'):
    try:
        __import__(_lib)
        print(f'✅ {_lib}')
    except Exception as _e:
        print(f'❌ {_lib} не встановлено — {_e}\n   Перезапусти Крок 1.')

if os.path.isdir('/content/CodeFormer'):
    print('✅ CodeFormer')
else:
    print('⚠️  CodeFormer не знайдено — портретний режим буде недоступний.')

try:
    _, _, _fd = shutil.disk_usage('/content')
    _fd_gb = _fd / 1073741824
    if _fd_gb < 1:
        print(f'⚠️  Диск вільно: {_fd_gb:.1f} GB — може не вистачити.')
    else:
        print(f'✅ Диск вільно: {_fd_gb:.1f} GB')
except Exception:
    pass

print()

# ════════════════════════════════════════════════
# §1  Google Drive (необов'язково)
# ════════════════════════════════════════════════
TEMP_DIR = '/content/photoenhance_temp'
os.makedirs(TEMP_DIR, exist_ok=True)

if os.path.isdir('/content/drive/MyDrive'):
    OUTPUT_DIR      = '/content/drive/MyDrive/PhotoEnhance_Results'
    DRIVE_AVAILABLE = True
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f'✅ Drive вже підключено → {OUTPUT_DIR}')
else:
    try:
        from google.colab import drive as _drv  # type: ignore
        print('🔗 Підключення Google Drive...')
        _drv.mount('/content/drive')
        OUTPUT_DIR      = '/content/drive/MyDrive/PhotoEnhance_Results'
        DRIVE_AVAILABLE = True
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        print(f'✅ Drive підключено → {OUTPUT_DIR}')
    except Exception as _e:
        print(f'⚠️  Drive: {_e}\n   Файл скачається напряму у Кроці 3.')
        OUTPUT_DIR      = TEMP_DIR
        DRIVE_AVAILABLE = False

MAX_FILE_MB = 100

# ════════════════════════════════════════════════
# §2  Стилі (адаптив для мобільних)
# ════════════════════════════════════════════════
display(HTML("""
<style>
.pe-info  {font-family:monospace;background:#f0f4ff;color:#1a1a2e;padding:10px;border-radius:6px;
           border-left:4px solid #4a90e2;font-size:13px;word-break:break-word;}
.pe-error {font-family:sans-serif;background:#fff0f0;padding:10px;border-radius:6px;
           border-left:4px solid #e74c3c;font-size:13px;}
@media(max-width:600px){
  .widget-upload,.widget-button{width:100%!important;min-height:52px!important;font-size:16px!important;}
  .widget-toggle-buttons .widget-toggle-button{font-size:11px!important;padding:4px 6px!important;}
}
</style>
"""))

# ════════════════════════════════════════════════
# §3  Інтерфейс
# ════════════════════════════════════════════════
_title = widgets.HTML(f"""
<div style="font-family:sans-serif;">
  <h2 style="margin:0 0 4px;">🎨 PhotoEnhance MVP</h2>
  <p style="color:#666;margin:0;">Real-ESRGAN + CodeFormer | Ліміт: <b>{MAX_FILE_MB} MB</b> | JPG/PNG/WEBP</p>
</div>
""")

upload_btn = widgets.FileUpload(
    accept='.jpg,.jpeg,.png,.webp',
    multiple=False,
    description='📸 Вибрати фото',
    layout=widgets.Layout(width='auto', min_width='200px', min_height='44px'),
)

photo_type = widgets.ToggleButtons(
    options=[
        ('🧑 Портрет',          'portrait'),
        ('👟 Товар / Текстура', 'product'),
        ('🌄 Пейзаж',          'landscape'),
        ('📷 Старе фото',       'old_photo'),
        ('🎌 Аніме',            'anime'),
    ],
    value='product',
    description='Тип:',
    style={'button_width': 'auto', 'description_width': '50px'},
    layout=widgets.Layout(width='100%'),
)

_type_desc = widgets.HTML("""
<div style="font-family:sans-serif;font-size:12px;color:#1a1a2e;
            background:#e8f0fe;border-left:4px solid #4a90e2;
            border-radius:6px;padding:10px 14px;margin-top:4px;line-height:1.7;">
  <b>🧑 Портрет</b> — CodeFormer: відновлює обличчя, зберігає природну шкіру.<br>
  <b>👟 Товар</b> — Real-ESRGAN: різкі деталі тканини, шкіри, гуми.<br>
  <b>🌄 Пейзаж</b> — Real-ESRGAN: чіткі контури та деталі фону.<br>
  <b>📷 Старе фото</b> — CodeFormer (обличчя) + ESRGAN (фон): усуває деградації.<br>
  <b>🎌 Аніме</b> — anime-6B модель: чисті лінії та кольори.
</div>
""")

face_fidelity = widgets.FloatSlider(
    value=0.8, min=0.0, max=1.0, step=0.05,
    description='Fidelity:',
    readout_format='.2f',
    style={'description_width': '70px'},
    layout=widgets.Layout(width='100%', max_width='480px'),
)
_fid_desc = widgets.HTML(
    '<div style="font-size:11px;color:#666;margin-top:-6px;">'
    '0.0 = макс відновлення &nbsp;|&nbsp; 0.8 = природна шкіра ✅ &nbsp;|&nbsp; 1.0 = майже без змін'
    '</div>'
)
_fid_box = widgets.VBox([face_fidelity, _fid_desc])
_fid_box.layout.display = 'none'

def _on_type_change(change):
    _fid_box.layout.display = '' if change['new'] in ('portrait', 'old_photo') else 'none'

photo_type.observe(_on_type_change, names='value')

#  Підказка користувача 
user_prompt = widgets.Textarea(
    value='',
    placeholder='Наприклад: прибрати шум, збільшити різкість, відновити обличчя, зменшити зернистість',
    description='💬 Побажання:',
    rows=3,
    style={'description_width': '100px'},
    layout=widgets.Layout(width='100%', max_width='620px'),
)
_prompt_note = widgets.HTML(
    '<div style="font-size:11px;color:#666;margin-top:-4px;">'
    '💬 Текст стане інструкцією для Кроку 4 (InstructPix2Pix). '
    'Необов\'язково.'
    '</div>'
)
_prompt_box = widgets.VBox([user_prompt, _prompt_note])


scale_options = widgets.ToggleButtons(
    options=[('×2  (швидше)', 2), ('×4  (оптимально)', 4), ('×8  (max розмір)', 8)],
    value=4,
    description='Масштаб:',
    style={'button_width': 'auto', 'description_width': '70px'},
)

model_options = widgets.Dropdown(
    options=[
        ('🖼️  Загальні фото',          'RealESRGAN_x4plus'),
        ('🎌 Аніме / Ілюстрації',       'RealESRGAN_x4plus_anime_6B'),
        ('📷 Деградовані фото',          'ESRGAN_SRx4_DF2KOST'),
        ('⚡ Швидкий режим (×2)',        'RealESRGAN_x2plus'),
    ],
    value='RealESRGAN_x4plus',
    description='Модель:',
    layout=widgets.Layout(width='100%', max_width='440px'),
)
_model_note = widgets.HTML(
    '<div style="font-size:11px;color:#888;">⚠️ Для Портрет / Старе фото / Аніме — модель обирається автоматично.</div>'
)

_preview = widgets.Output()
_info    = widgets.HTML('<div class="pe-info">📁 Фото ще не вибрано</div>')

def _on_upload(*_):
    """Викликається при виборі файлу (observer callback, аргумент не потрібен)."""
    if not upload_btn.value:
        return
    for _fname, _fdata in upload_btn.value.items():
        _size_mb = len(_fdata['content']) / 1048576
        if _size_mb > MAX_FILE_MB:
            _info.value = (
                f'<div class="pe-error">❌ <b>Файл {_size_mb:.1f} MB</b> — '
                f'перевищує ліміт {MAX_FILE_MB} MB. Стисни або обери інший.</div>'
            )
            with _preview:
                clear_output()
            return

        try:
            _img = Image.open(io.BytesIO(_fdata['content']))
        except Exception as _e:
            _info.value = f'<div class="pe-error">❌ Неможливо відкрити файл: {_e}</div>'
            with _preview:
                clear_output()
            return

        _w, _h = _img.size
        _mpx   = _w * _h / 1_000_000
        _info.value = (
            f'<div class="pe-info">'
            f'✅ <b>{_fname}</b><br>'
            f'📐 {_w}×{_h} px &nbsp;|&nbsp; {_size_mb:.1f} MB &nbsp;|&nbsp;'
            f' {_img.mode} &nbsp;|&nbsp; {_mpx:.1f} Mpx'
            f'</div>'
        )
        with _preview:
            clear_output(wait=True)
            _thumb = _img.copy()
            _thumb.thumbnail((280, 280))
            _, _ax = plt.subplots(figsize=(3.5, 3.5))
            _ax.imshow(_thumb)
            _ax.set_title('Прев\'ю оригіналу', fontsize=10)
            _ax.axis('off')
            plt.tight_layout()
            plt.show()

upload_btn.observe(_on_upload, names='value')

_ui = widgets.VBox([
    _title,
    widgets.HTML('<hr style="margin:8px 0">'),
    upload_btn, _info, _preview,
    widgets.HTML('<hr style="margin:8px 0">'),
    photo_type, _type_desc, _fid_box,
    widgets.HTML('<hr style="margin:8px 0">'),
    model_options, _model_note, scale_options,
    widgets.HTML('<hr style="margin:8px 0">'),



    widgets.HTML('<hr style="margin:8px 0">'),
    _prompt_box,
    widgets.HTML('<b>✅ Готово? Запускай ▶ Крок 3</b>'),
], layout=widgets.Layout(width='100%', max_width='660px'))
display(_ui)

In [ ]:
# @title ⚡ Крок 3 — Обробка фото { display-mode: "form" }
# @markdown Запускай після вибору фото у Кроці 2. Можна запускати повторно — модель береться з кешу.

# ══════════════════════════════════════════════════════════
# АКТИВНІ ОПТИМІЗАЦІЇ ШВИДКОСТІ:
#
#  ⚡ FP16           — 4–8× швидший інференс (T4/A100)
#  🔥 TF32           — +20% на Ampere/Hopper без втрат якості
#  ⚙️ cudnn.bench    — авто-tune CUDA kernelів під поточне GPU
#  📐 Адаптив tile   — 0 (мале) / 512 / 256 → менше overhead
#  🚀 inference_mode — швидший за torch.no_grad
#  ♻️ Кеш моделей    — перший запуск: завантаження; далі: миттєво
#  🛠 torch.compile  — JIT-компіляція (PyTorch 2.0+, разовий warmup)
#  🛡 OOM fallback   — при нестачі VRAM авто-retry з tile=128
# ══════════════════════════════════════════════════════════

import os
import io
import re
import sys
import time
import base64
import shutil
import urllib.request
from types import ModuleType

import cv2
import torch
import ipywidgets as widgets
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
from IPython.display import display, HTML
from torchvision.transforms.functional import normalize as tv_normalize

# ── torchvision compat ──────────────────────────────────────────────
if 'torchvision.transforms.functional_tensor' not in sys.modules:
    import torchvision.transforms.functional as _tvf
    _cm = ModuleType('torchvision.transforms.functional_tensor')
    _cm.rgb_to_grayscale = _tvf.rgb_to_grayscale  # type: ignore
    sys.modules['torchvision.transforms.functional_tensor'] = _cm

cv2.setNumThreads(0)           # OpenCV: авто вибір потоків
Image.MAX_IMAGE_PIXELS = None  # PIL: не кидати помилку на великих фото

for _p in ('/content/Real-ESRGAN', '/content/CodeFormer'):
    if _p not in sys.path:
        sys.path.insert(0, _p)

# Кеш між повторними запусками клітинки
if '_PE_CACHE' not in globals():
    _PE_CACHE: dict = {}

# PyTorch оптимізації
torch.backends.cudnn.benchmark        = True
torch.backends.cudnn.allow_tf32       = True
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.set_float32_matmul_precision('high')
except AttributeError:
    pass

USE_FP16 = torch.cuda.is_available()
print(
    f'⚡ {"FP16 + TF32 + cudnn.bench" if USE_FP16 else "FP32 CPU (повільно)"}'
    f'  |  PyTorch {torch.__version__}'
    f'  |  CUDA {torch.version.cuda if USE_FP16 else "N/A"}'
)

# ════════════════════════════════════════════════
# §A  АДРЕСИ МОДЕЛЕЙ
# ════════════════════════════════════════════════
_MODEL_URLS: dict = {
    'RealESRGAN_x4plus': (
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth', 4),
    'RealESRGAN_x4plus_anime_6B': (
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_anime_6B.pth', 4),
    'RealESRGAN_x2plus': (
        'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth', 2),
    'ESRGAN_SRx4_DF2KOST': (
        'https://github.com/xinntao/ESRGAN/releases/download/v0.1.1/ESRGAN_SRx4_DF2KOST_official-ff704c30.pth', 4),
}
_MODELS_DIR = '/content/pe_models'
os.makedirs(_MODELS_DIR, exist_ok=True)

# ════════════════════════════════════════════════
# §B  ДОПОМІЖНІ ФУНКЦІЇ
# ════════════════════════════════════════════════
def _download_model(name: str) -> str:
    """Завантажує вагу моделі якщо відсутня, повертає шлях."""
    model_url, _ = _MODEL_URLS[name]
    path = os.path.join(_MODELS_DIR, f'{name}.pth')
    if not os.path.isfile(path):
        print(f'⬇️  Завантаження {name}...')
        urllib.request.urlretrieve(model_url, path)
        print(f'✅ {name} збережено')
    else:
        print(f'♻️  {name} вже є')
    return path


def _adaptive_tile(h: int, w: int) -> int:
    """Адаптивний розмір тайла: 0 / 512 / 256 залежно від Mpx."""
    mpx = h * w / 1_000_000
    if   mpx < 2: return 0
    elif mpx < 8: return 512
    else:         return 256


def _warmup(model: torch.nn.Module, device: torch.device, fp16: bool) -> None:
    """Warmup: прогрів CUDA kernelів і torch.compile перед першим фото."""
    try:
        d = torch.zeros(1, 3, 64, 64, device=device)
        if fp16:
            d = d.half()
        with torch.inference_mode():
            model(d)
        if device.type == 'cuda':
            torch.cuda.synchronize()
        print('🔥 Warmup OK')
    except Exception:
        pass  # деякі мережі не приймають довільний вхід


def _build_esrgan(name: str, img_h: int = 0, img_w: int = 0):
    """Будує або повертає з кешу RealESRGANer зі всіма оптимізаціями."""
    key = f'esrgan_{name}_{USE_FP16}'
    if key in _PE_CACHE:
        up = _PE_CACHE[key]
        t = _adaptive_tile(img_h, img_w)
        up.tile = t; up.tile_pad = 32 if t else 0
        print(f'♻️  ESRGAN кеш: {name}  tile={t}')
        return up

    from realesrgan import RealESRGANer
    from basicsr.archs.rrdbnet_arch import RRDBNet

    _, scale = _MODEL_URLS[name]      # _ = url (не потрібен тут)
    path     = _download_model(name)
    n_block  = 6 if 'anime_6B' in name else 23

    rrd = RRDBNet(
        num_in_ch=3, num_out_ch=3, num_feat=64,
        num_block=n_block, num_grow_ch=32, scale=scale,
    )
    tile = _adaptive_tile(img_h, img_w); tile_pad = 32 if tile else 0

    up = RealESRGANer(
        scale=scale, model_path=path, model=rrd,
        tile=tile, tile_pad=tile_pad, pre_pad=0,
        half=USE_FP16,
    )
    up.model.eval()

    # torch.compile removed: conflicts with tile-loop CUDAGraphs

    _dev = torch.device('cuda' if USE_FP16 else 'cpu')
    _warmup(up.model, _dev, USE_FP16)

    _PE_CACHE[key] = up
    print(f'✅ ESRGAN: {name}  tile={tile}  FP{"16" if USE_FP16 else "32"}')
    return up


def _build_codeformer():
    """Завантажує або повертає з кешу CodeFormer."""
    key = f'codeformer_{USE_FP16}'
    if key in _PE_CACHE:
        print('♻️  CodeFormer кеш')
        return _PE_CACHE[key]

    import importlib
    importlib.import_module('basicsr.archs.codeformer_arch')
    from basicsr.utils.registry      import ARCH_REGISTRY
    from basicsr.utils.download_util import load_file_from_url

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    net = ARCH_REGISTRY.get('CodeFormer')(
        dim_embd=512, codebook_size=1024, n_head=8, n_layers=9,
        connect_list=['32', '64', '128', '256'],
    ).to(device)

    ckpt = load_file_from_url(
        url='https://github.com/sczhou/CodeFormer/releases/download/v0.1.0/codeformer.pth',
        model_dir='/content/CodeFormer/weights/CodeFormer',
        progress=True, file_name=None,
    )
    net.load_state_dict(torch.load(ckpt, map_location=device)['params_ema'])
    net.eval()
    if USE_FP16:
        net = net.half()

    if not USE_FP16:  # torch.compile + FP16 => Dynamo dtype mismatch
        try:
            net = torch.compile(net, mode='reduce-overhead')
            print('⚡ torch.compile: CodeFormer')
        except Exception:
            pass

    result = (net, device)
    _PE_CACHE[key] = result
    print('✅ CodeFormer готовий')
    return result


def _resolve_model(photo_type: str, model_name: str, scale: int):
    if photo_type == 'anime':
        return 'RealESRGAN_x4plus_anime_6B', min(scale, 4)
    if photo_type in ('portrait', 'old_photo'):
        return 'RealESRGAN_x4plus', scale
    return model_name, scale

# ════════════════════════════════════════════════
# §C  СТАНДАРТНИЙ PIPELINE
# ════════════════════════════════════════════════
def enhance_standard(
    input_path: str, output_path: str,
    model_name: str, outscale: int,
    pb: widgets.IntProgress,
) -> dict:
    img = cv2.imread(input_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise ValueError(f'Не вдалося прочитати: {input_path}')
    h, w = img.shape[:2]
    tile = _adaptive_tile(h, w)
    print(f'📐 {w}×{h} px  |  {model_name}  |  ×{outscale}  |  tile={tile or "вимк"}')
    pb.value = 10; pb.description = '⬇️  10%'
    up = _build_esrgan(model_name, h, w)

    _real = sys.stdout

    class _TileProgress:
        def write(self, text: str) -> None:
            m = re.search(r'Tile\s+(\d+)/(\d+)', text)
            if m:
                cur, tot = int(m.group(1)), int(m.group(2))
                pct = 15 + int(cur / tot * 80)
                pb.value = min(pct, 95); pb.description = f'🔄 {min(pct, 95):>3}%'
        def flush(self) -> None: _real.flush()
        def __getattr__(self, n: str): return getattr(_real, n)

    sys.stdout = _TileProgress()
    try:
        try:
            torch.compiler.cudagraph_mark_step_begin()
        except Exception:
            pass
        out_img, _ = up.enhance(img, outscale=outscale)
    except RuntimeError as exc:
        sys.stdout = _real
        if 'out of memory' in str(exc).lower():
            torch.cuda.empty_cache()
            up.tile = 128; up.tile_pad = 16
            print('⚠️  OOM — retry tile=128...')
            sys.stdout = _TileProgress()
            try:
                torch.compiler.cudagraph_mark_step_begin()
            except Exception:
                pass
            out_img, _ = up.enhance(img, outscale=outscale)
            sys.stdout = _real
        else:
            raise
    except Exception:
        sys.stdout = _real
        raise
    else:
        sys.stdout = _real

    if USE_FP16:
        torch.cuda.synchronize()
    h_out, w_out = out_img.shape[:2]
    _fl = [cv2.IMWRITE_JPEG_QUALITY, 95] if output_path.lower().endswith(('.jpg', '.jpeg')) else []
    cv2.imwrite(output_path, out_img, _fl)
    del out_img; torch.cuda.empty_cache()
    return {'orig': (w, h), 'out': (w_out, h_out)}

# ════════════════════════════════════════════════
# §D  ПОРТРЕТ / СТАРЕ ФОТО (CodeFormer + ESRGAN)
# ════════════════════════════════════════════════
def enhance_portrait(
    input_path: str, output_path: str,
    outscale: int, fidelity: float,
    pb: widgets.IntProgress,
) -> dict:
    from basicsr.utils import img2tensor, tensor2img
    from facelib.utils.face_restoration_helper import FaceRestoreHelper

    img = cv2.imread(input_path, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f'Не вдалося прочитати: {input_path}')
    h, w = img.shape[:2]
    print(f'📐 {w}×{h} px  |  CodeFormer  fidelity={fidelity}  ×{outscale}')

    pb.value = 5;  pb.description = '⬇️   5%'
    net, device = _build_codeformer()
    pb.value = 18; pb.description = '🔬  18%'
    bg_up = _build_esrgan('RealESRGAN_x4plus', h, w)
    pb.value = 28; pb.description = '⬇️  28%'

    helper = FaceRestoreHelper(
        outscale, face_size=512, crop_ratio=(1, 1),
        det_model='retinaface_resnet50',
        save_ext='png', use_parse=True, device=device,
    )
    helper.clean_all()
    helper.read_image(img)
    n_faces = helper.get_face_landmarks_5(
        only_center_face=False, resize=640, eye_dist_threshold=5,
    )
    print(f'🧑 Знайдено облич: {n_faces}')

    if n_faces == 0:
        print('⚠️  Обличчя не виявлено — ESRGAN без CodeFormer.')
        pb.value = 32
        return enhance_standard(input_path, output_path, 'RealESRGAN_x4plus', outscale, pb)

    helper.align_warp_face()
    pb.value = 38; pb.description = '🔬  38%'
    n_total = len(helper.cropped_faces)

    for idx, cropped in enumerate(helper.cropped_faces):
        cf_t = img2tensor(cropped / 255.0, bgr2rgb=True, float32=True)
        tv_normalize(cf_t, (0.5, 0.5, 0.5), (0.5, 0.5, 0.5), inplace=True)
        cf_t = cf_t.unsqueeze(0).to(device)
        if USE_FP16:
            cf_t = cf_t.half()
        try:
            with torch.inference_mode():
                restored_t = net(cf_t, w=fidelity, adain=True)[0]
            restored = tensor2img(restored_t.float(), rgb2bgr=True, min_max=(-1, 1))
            del restored_t; torch.cuda.empty_cache()
        except Exception as exc:
            print(f'⚠️  Обличчя {idx + 1}: {exc} — залишаю оригінал.')
            restored = tensor2img(cf_t.float().squeeze(0), rgb2bgr=True, min_max=(-1, 1))
        helper.add_restored_face(restored.astype('uint8'))
        pct = 38 + int((idx + 1) / n_total * 35)
        pb.value = pct; pb.description = f'🔬 {pct:>3}%'

    pb.value = 75; pb.description = '🌄  75%'
    helper.face_upsampler = bg_up
    try:
        torch.compiler.cudagraph_mark_step_begin()
    except Exception:
        pass
    bg_img = bg_up.enhance(img, outscale=outscale)[0]
    pb.value = 90; pb.description = '🔧  90%'

    try:
        helper.get_inverse_affine(None)
    except TypeError:
        helper.get_inverse_affine(save_inverse_affine_input_size=False)

    result_img = helper.paste_faces_to_input_image(upsample_img=bg_img, draw_box=False)
    h_out, w_out = result_img.shape[:2]
    _fl = [cv2.IMWRITE_JPEG_QUALITY, 95] if output_path.lower().endswith(('.jpg', '.jpeg')) else []
    cv2.imwrite(output_path, result_img, _fl)
    del result_img, bg_img; torch.cuda.empty_cache()
    return {'orig': (w, h), 'out': (w_out, h_out)}

# ════════════════════════════════════════════════
# §E  ПЕРЕВІРКИ ПЕРЕД ЗАПУСКОМ
# ════════════════════════════════════════════════
def _abort(msg: str) -> None:
    raise SystemExit(f'❌ {msg}')

# E-1 попередні кроки
for _var, _step in (('torch', 'Крок 1'), ('upload_btn', 'Крок 2'), ('OUTPUT_DIR', 'Крок 2')):
    if _var not in globals():
        _abort(f'{_step} не виконано — запускай кроки по порядку.')
print('✅ Кроки 1 і 2 виконано')

# E-2 файл вибрано
if not upload_btn.value:  # type: ignore
    _abort('Фото не вибрано — поверніся до Кроку 2 і вибери файл.')

_fname   = list(upload_btn.value.keys())[0]    # type: ignore
_fraw    = list(upload_btn.value.values())[0]  # type: ignore
_size_mb = len(_fraw['content']) / 1048576
print(f'✅ Файл: {_fname}  ({_size_mb:.1f} MB)')

_max_mb = globals().get('MAX_FILE_MB', 100)
if _size_mb > _max_mb:
    _abort(f'Файл {_size_mb:.1f} MB > ліміту {_max_mb} MB.')
if _size_mb == 0:
    _abort('Файл порожній.')

# E-3 розмір зображення — ініціалізуємо до try (щоб точно bound)
_img_w, _img_h = 0, 0
try:
    _pil = Image.open(io.BytesIO(_fraw['content']))
    _img_w, _img_h = _pil.size
    print(f'✅ {_img_w}×{_img_h} px  ({_img_w * _img_h / 1e6:.1f} Mpx)  |  {_pil.mode}')
except Exception as exc:
    _abort(f'Не вдалося відкрити: {exc}')

_sel_scale  = scale_options.value if 'scale_options' in globals() else 4    # type: ignore
_photo_type = photo_type.value    if 'photo_type'    in globals() else 'product'  # type: ignore
_fidelity   = face_fidelity.value if 'face_fidelity' in globals() else 0.8  # type: ignore

_out_mpx = _img_w * _img_h * _sel_scale ** 2 / 1_000_000
if _out_mpx > 150:
    _abort(
        f'❌ Вихід {_out_mpx:.0f} Mpx — завелике! '
        f'Макс вхід при ×{_fin_scale}: {int((150/_fin_scale**2)**0.5)}×{int((150/_fin_scale**2)**0.5)} px '
        f'({150//_fin_scale**2} Mpx). Зменш масштаб або обріж фото.'
    )
elif _out_mpx > 80:
    print(f'⚠️  Вихід {_out_mpx:.0f} Mpx → JPEG q=95 | макс вхід при ×{_fin_scale}: {int((150/_fin_scale**2)**0.5)}×{int((150/_fin_scale**2)**0.5)} px')
else:
    print(f'✅ Вихід ~{_out_mpx:.0f} Mpx → PNG | макс вхід при ×{_fin_scale}: PNG: {int((80/_fin_scale**2)**0.5)}×{int((80/_fin_scale**2)**0.5)} px / JPEG: {int((150/_fin_scale**2)**0.5)}×{int((150/_fin_scale**2)**0.5)} px')

# E-4 VRAM
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    _free_gb = (
        torch.cuda.get_device_properties(0).total_memory
        - torch.cuda.memory_allocated()
    ) / 1073741824
    if _free_gb < 1.5:
        _abort(f'Мало VRAM ({_free_gb:.1f} GB). Runtime → Restart runtime.')
    print(f'✅ Вільна VRAM: {_free_gb:.1f} GB')
else:
    print('⚠️  GPU відсутній — CPU (дуже повільно).')

# E-5 диск
try:
    _, _, _fd = shutil.disk_usage('/content')
    if _fd / 1073741824 < 0.5:
        _abort(f'Диск: лише {_fd / 1073741824:.1f} GB вільно.')
    print(f'✅ Диск: {_fd / 1073741824:.1f} GB вільно')
except Exception:
    pass

# E-6 CodeFormer
if _photo_type in ('portrait', 'old_photo'):
    if not os.path.isdir('/content/CodeFormer'):
        _abort('CodeFormer не знайдено — перезапусти Крок 1.')
    print('✅ CodeFormer доступний')

# ════════════════════════════════════════════════
# §F  ЗАПУСК
# ════════════════════════════════════════════════
from google.colab import files as _colab_files  # type: ignore  # noqa: F401

_LABELS = {
    'portrait': '🧑 Портрет', 'product': '👟 Товар',
    'landscape': '🌄 Пейзаж', 'old_photo': '📷 Старе фото', 'anime': '🎌 Аніме',
}
_sel_model  = model_options.value if 'model_options' in globals() else 'RealESRGAN_x4plus'  # type: ignore
_fin_model, _fin_scale = _resolve_model(_photo_type, _sel_model, _sel_scale)
_fp_lbl    = 'FP16+TF32' if USE_FP16 else 'FP32'
_type_lbl  = _LABELS.get(_photo_type, _photo_type)

TEMP_DIR = '/content/photoenhance_temp'
os.makedirs(TEMP_DIR, exist_ok=True)

_base     = os.path.splitext(_fname)[0]
_in_path  = os.path.join(TEMP_DIR, _fname)
_out_ext  = 'jpg' if _out_mpx > 80 else 'png'
_out_name = f'{_base}_x{_fin_scale}.{_out_ext}'
_out_path = os.path.join(TEMP_DIR, _out_name)
_drv_path = os.path.join(OUTPUT_DIR, _out_name)  # type: ignore

with open(_in_path, 'wb') as _fh:
    _fh.write(_fraw['content'])

print('=' * 58)
print(f'🎨 PhotoEnhance  |  {_fname}')
if _photo_type in ('portrait', 'old_photo'):
    print(f'   CodeFormer (fidelity={_fidelity}) + ESRGAN фон  ×{_fin_scale}  {_fp_lbl}')
else:
    print(f'   {_fin_model}  ×{_fin_scale}  {_fp_lbl}  adap.tile  warmup')
print('=' * 58)

_pb = widgets.IntProgress(
    value=0, min=0, max=100,
    description='⏳  0%', bar_style='info',
    style={'bar_color': '#4a90e2', 'description_width': '65px'},
    layout=widgets.Layout(width='100%', height='28px'),
)
display(_pb)

_t0 = time.time()
try:
    if _photo_type in ('portrait', 'old_photo'):
        _stats = enhance_portrait(_in_path, _out_path, _fin_scale, _fidelity, _pb)
    else:
        _stats = enhance_standard(_in_path, _out_path, _fin_model, _fin_scale, _pb)
    _pb.value = 100; _pb.description = '✅ 100%'; _pb.bar_style = 'success'
except Exception:
    _pb.bar_style = 'danger'; _pb.description = '❌ Помилка'
    raise

_elapsed = time.time() - _t0
print(f'\n✅ {_elapsed:.1f} сек  →  {_stats["out"][0]}×{_stats["out"][1]} px')

if 'DRIVE_AVAILABLE' in globals() and DRIVE_AVAILABLE and _drv_path != _out_path:  # type: ignore
    shutil.copy2(_out_path, _drv_path)
    print(f'☁️  Drive: {_drv_path}')

# ════════════════════════════════════════════════
# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═
# §G  BEFORE / AFTER — інтерактивний слайдер
# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═# ═
import io as _io
_orig_pil = Image.open(_in_path).convert('RGB')
_enh_pil  = Image.open(_out_path).convert('RGB')
_MAX_D = 1200
_orig_show = _orig_pil.copy(); _orig_show.thumbnail((_MAX_D, _MAX_D), Image.LANCZOS)
_enh_show  = _enh_pil.copy();  _enh_show.thumbnail((_MAX_D, _MAX_D),  Image.LANCZOS)

def _pil_b64(img):
    buf = _io.BytesIO()
    img.save(buf, format='JPEG', quality=88)
    return base64.b64encode(buf.getvalue()).decode()

_b64_orig = _pil_b64(_orig_show)
_b64_enh  = _pil_b64(_enh_show)
# зчитуємо побажання користувача (з Кроку 2)
_user_note_html = ''
try:
    _up_text = user_prompt.value.strip()
    if _up_text:
        import html as _html_mod
        _up_esc = _html_mod.escape(_up_text)
        _user_note_html = (
            '<div style="font-family:sans-serif;font-size:12px;'
            'background:#fff8e1;border-left:4px solid #f9a825;'
            'border-radius:6px;padding:8px 12px;margin-top:8px;'
            'color:#4a3800;">'
            f'<b> Побажання:</b> {_up_esc}'
            '</div>'
        )
except Exception:
    pass


display(HTML(f'''
<div style="font-family:sans-serif;max-width:700px;user-select:none;">
  <p style="color:#555;font-size:13px;margin:4px 0 6px;">
    ← Тягни слайдер або перетягуй по фото для порівняння BEFORE / AFTER →
  </p>
  <div id="pe-cmp" style="position:relative;display:block;width:100%;max-width:700px;
       cursor:col-resize;border-radius:8px;overflow:hidden;box-shadow:0 2px 8px rgba(0,0,0,.3);">
    <img src="data:image/jpeg;base64,{_b64_enh}" style="display:block;width:100%;height:auto;">
    <div id="pe-clip" style="position:absolute;top:0;left:0;width:50%;height:100%;overflow:hidden;">
      <img src="data:image/jpeg;base64,{_b64_orig}" style="width:200%;max-width:none;height:auto;display:block;">
    </div>
    <div id="pe-line" style="position:absolute;top:0;left:50%;width:3px;height:100%;
         background:#fff;box-shadow:0 0 6px rgba(0,0,0,.5);">
      <div style="position:absolute;top:50%;left:50%;transform:translate(-50%,-50%);
           background:#fff;border-radius:50%;width:34px;height:34px;
           display:flex;align-items:center;justify-content:center;
           box-shadow:0 0 6px rgba(0,0,0,.4);font-size:18px;color:#333;">⇔</div>
    </div>
    <div style="position:absolute;bottom:8px;left:8px;background:rgba(231,76,60,.85);
         color:#fff;padding:3px 10px;border-radius:4px;font-size:12px;font-weight:bold;">BEFORE</div>
    <div style="position:absolute;bottom:8px;right:8px;background:rgba(46,204,113,.85);
         color:#fff;padding:3px 10px;border-radius:4px;font-size:12px;font-weight:bold;">AFTER x{_fin_scale}</div>
  </div>
  <input type="range" min="0" max="100" value="50" id="pe-slider"
         style="width:100%;max-width:700px;margin-top:8px;accent-color:#4a90e2;">
</div>
<script>
(function(){{
  var c=document.getElementById('pe-cmp');
  var clip=document.getElementById('pe-clip');
  var line=document.getElementById('pe-line');
  var sl=document.getElementById('pe-slider');
  function set(p){{
    p=Math.max(0,Math.min(100,p));
    clip.style.width=p+'%';
    line.style.left=p+'%';
    sl.value=p;
  }}
  sl.addEventListener('input',function(){{set(+this.value);}});
  var dragging=false;
  function pos(e){{
    var r=c.getBoundingClientRect();
    var x=(e.touches?e.touches[0].clientX:e.clientX)-r.left;
    set(x/r.width*100);
  }}
  c.addEventListener('mousedown',function(e){{dragging=true;pos(e);}});
  document.addEventListener('mouseup',function(){{dragging=false;}});
  document.addEventListener('mousemove',function(e){{if(dragging)pos(e);}});
  c.addEventListener('touchstart',function(e){{pos(e);}},{{passive:true}});
  c.addEventListener('touchmove',function(e){{pos(e);e.preventDefault();}},{{passive:false}});
}})();
</script>'''))

if _user_note_html:
    display(HTML(_user_note_html))

# side-by-side JPEG for download
_cmp_path = os.path.join(TEMP_DIR, 'comparison.jpg')
_cw = _orig_show.width + _enh_show.width
_ch = max(_orig_show.height, _enh_show.height)
_cmp_img = Image.new('RGB', (_cw, _ch), (26, 26, 46))
_cmp_img.paste(_orig_show, (0, 0))
_cmp_img.paste(_enh_show, (_orig_show.width, 0))
_cmp_img.save(_cmp_path, 'JPEG', quality=92)

_px_x = (_stats['out'][0] * _stats['out'][1]) / max(1, _stats['orig'][0] * _stats['orig'][1])
print('📊 СТАТИСТИКА')
print(f'  📥 {_stats["orig"][0]}×{_stats["orig"][1]} px  →  📤 {_stats["out"][0]}×{_stats["out"][1]} px')
print(f'  🔍 ×{_fin_scale}  |  📈 пікселів ×{_px_x:.1f}  |  ⏱️  {_elapsed:.1f} сек  |  {_fp_lbl}')
    print(f'  🔬 CodeFormer fidelity={_fidelity}  |  {_fin_model} (фон)')
    print(f'  🤖 {_fin_model}')
# §H  СКАЧУВАННЯ
# ════════════════════════════════════════════════
def _dl_link(path: str, label: str, mime: str = 'image/png') -> str:
    with open(path, 'rb') as fh:
        b64 = base64.b64encode(fh.read()).decode()
    fn = os.path.basename(path)
    return (
        f'<a href="data:{mime};base64,{b64}" download="{fn}" '
        f'style="display:inline-block;margin:6px 4px;padding:11px 18px;'
        f'background:#2ecc71;color:#fff;font-weight:bold;font-size:14px;'
        f'border-radius:8px;text-decoration:none;min-width:180px;text-align:center;">'
        f'⬇️ {label}</a>'
    )

display(HTML(
    '<div style="background:#f0f4ff;color:#1a1a2e;border-radius:8px;padding:10px;margin-top:8px;'
    'border-left:4px solid #4a90e2;font-family:sans-serif;">'
    '<b>📱 На телефоні:</b> кнопка нижче → браузер збереже файл.<br>'
    '<small>Не спрацювало — утримуй зображення → "Зберегти".</small>'
    '</div>'
    + _dl_link(_out_path, 'Покращене фото')
    + _dl_link(_cmp_path, 'Before / After')
))

try:
    _colab_files.download(_out_path)
    _colab_files.download(_cmp_path)
except Exception:
    pass

print('\n✅ Готово!')


In [ ]:
# @title 🔍 ДІАГНОСТИКА — запусти та скопіюй весь вивід
import subprocess, sys, os

print("=" * 60)
print("1) basicsr версія та шлях:")
r = subprocess.run(['pip', 'show', 'basicsr'], capture_output=True, text=True)
print(r.stdout.strip())

print("\n2) realesrgan версія та шлях:")
r2 = subprocess.run(['pip', 'show', 'realesrgan'], capture_output=True, text=True)
print(r2.stdout.strip())

print("\n3) Де Python знаходить basicsr:")
try:
    import basicsr
    bpath = os.path.dirname(basicsr.__file__)
    print(f"  {bpath}")
    print("\n4) Файли у basicsr/data/:")
    data_path = os.path.join(bpath, 'data')
    if os.path.isdir(data_path):
        for f in sorted(os.listdir(data_path)):
            print(f"  {f}")
    else:
        print("  ⚠️  Папка basicsr/data НЕ існує!")
except Exception as e:
    print(f"  ❌ import basicsr: {e}")

print("\n5) sys.path (перші 10):")
for p in sys.path[:10]:
    print(f"  {p}")

print("\n6) Де Python знаходить realesrgan:")
try:
    import importlib, importlib.util
    spec = importlib.util.find_spec('realesrgan')
    print(f"  {spec.origin if spec else 'не знайдено'}")
    print("\n7) Вміст realesrgan/__init__.py (перші 30 рядків):")
    init = os.path.join(os.path.dirname(spec.origin), '__init__.py') if spec else None
    if init and os.path.isfile(init):
        with open(init) as f:
            for i, line in enumerate(f):
                if i >= 30: break
                print(f"  {line}", end='')
    else:
        print("  не знайдено")
except Exception as e:
    print(f"  ❌ {e}")

print("\n" + "=" * 60)
print("Скопіюй весь цей вивід — це дасть точну відповідь!")


---
## ✨ Крок 4 — InstructPix2Pix (текстові інструкції)

**Запускай після Кроку 3.** Бере покращене фото з Кроку 3 і змінює його згідно з інструкцією з поля **«Побажання»** (Крок 2).

**Приклади інструкцій (англійською — краща якість):**
- `make the sky more blue`
- `remove noise and sharpen details`
- `make it look like a professional photo`
- `add warm sunset lighting`
- `make colors more vivid and saturated`
- `convert to black and white`

> ⚠️ Перший запуск завантажує ~4 GB модель — зачекай. Далі береться з кешу.
> IP2P оптимально працює на 512 px — фото автоматично масштабується.


In [ ]:
# @title ✨ Крок 4 — InstructPix2Pix { display-mode: "form" }
# @markdown Запускай після Кроку 3. Поле "💬 Побажання" з Кроку 2 стає інструкцією.
# @markdown Приклад: "make the sky more blue", "remove noise", "add warm lighting"

import sys, os, io, base64, time as _time
import torch
import html as _html_mod
from PIL import Image
from IPython.display import display, HTML

# ── Перевірки ───────────────────────────────────────────────────
if '_out_path' not in dir() or not os.path.isfile(_out_path):
    raise SystemExit('❌ Спочатку запусти Крок 3!')

try:
    _ip2p_prompt = user_prompt.value.strip()
except Exception:
    _ip2p_prompt = ''

if not _ip2p_prompt:
    raise SystemExit(
        '❌ Поле "💬 Побажання" у Кроці 2 порожнє!\n'
        '   Введи інструкцію англійською, наприклад:\n'
        '   "make the photo sharper and more vivid"'
    )

print(f'💬 Інструкція: {_ip2p_prompt}')

# ── Встановлення diffusers ────────────────────────────────────
try:
    import diffusers as _diff
    print(f'✅ diffusers {_diff.__version__}')
except ImportError:
    print('📦 Встановлення diffusers...')
    import subprocess
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'diffusers>=0.25', 'transformers', 'accelerate'],
        check=True
    )
    import diffusers as _diff
    print(f'✅ diffusers {_diff.__version__}')

from diffusers import StableDiffusionInstructPix2PixPipeline

# ── Параметри ───────────────────────────────────────────────────
IP2P_STEPS   = 50    # 20=швидше/гірше, 75=краще/повільніше
IP2P_TEXT_GS = 7.5   # text guidance scale
IP2P_IMG_GS  = 1.5   # image guidance: 1.0=більше змін, 2.5=менше змін

# ── Завантаження моделі ────────────────────────────────────────
_ip2p_key = 'ip2p_pipe'
if _ip2p_key not in _PE_CACHE:
    print('⬇️  Завантаження InstructPix2Pix (~4 GB, перший раз)...')
    _ip2p_dev   = 'cuda' if torch.cuda.is_available() else 'cpu'
    _ip2p_dtype = torch.float16 if _ip2p_dev == 'cuda' else torch.float32
    _pipe = StableDiffusionInstructPix2PixPipeline.from_pretrained(
        'timbrooks/instruct-pix2pix',
        torch_dtype=_ip2p_dtype,
        safety_checker=None,
    ).to(_ip2p_dev)
    _pipe.enable_attention_slicing()
    _PE_CACHE[_ip2p_key] = _pipe
    print('✅ Модель завантажена')
else:
    _pipe = _PE_CACHE[_ip2p_key]
    print('♻️  Модель з кешу')

# ── Підготовка зображення ──────────────────────────────────────
_MAX_IP2P   = 512
_in_img_all = Image.open(_out_path).convert('RGB')
_in_img     = _in_img_all.copy()
if max(_in_img.size) > _MAX_IP2P:
    _in_img.thumbnail((_MAX_IP2P, _MAX_IP2P), Image.LANCZOS)
    print(f'📐 Масштабовано до {_in_img.size[0]}×{_in_img.size[1]} px для IP2P')

# ── Інференс ──────────────────────────────────────────────────
print(f'🔄 Обробка ({IP2P_STEPS} кроків)...')
_t0 = _time.time()
with torch.inference_mode():
    _ip2p_result = _pipe(
        prompt=_ip2p_prompt,
        image=_in_img,
        num_inference_steps=IP2P_STEPS,
        guidance_scale=IP2P_TEXT_GS,
        image_guidance_scale=IP2P_IMG_GS,
    ).images[0]
_ip2p_elapsed = _time.time() - _t0
print(f'✅ Готово за {_ip2p_elapsed:.1f} сек')

# ── Зберегти результат ──────────────────────────────────────────
_ip2p_out = os.path.join(TEMP_DIR, 'ip2p_result.jpg')
_ip2p_result.save(_ip2p_out, 'JPEG', quality=92)

# ── Слайдер Before/After ─────────────────────────────────────────
def _ip2p_b64j(img):
    _b = io.BytesIO()
    img.save(_b, 'JPEG', quality=88)
    return base64.b64encode(_b.getvalue()).decode()

_MAX_D    = 800
_ip2p_bef = _in_img.copy();      _ip2p_bef.thumbnail((_MAX_D, _MAX_D), Image.LANCZOS)
_ip2p_aft = _ip2p_result.copy(); _ip2p_aft.thumbnail((_MAX_D, _MAX_D), Image.LANCZOS)
_b64_bef  = _ip2p_b64j(_ip2p_bef)
_b64_aft  = _ip2p_b64j(_ip2p_aft)
_prompt_esc = _html_mod.escape(_ip2p_prompt)

display(HTML(
    '<h3 style="font-family:sans-serif;margin:16px 0 4px;">✨ InstructPix2Pix результат</h3>'
    '<p style="font-family:sans-serif;font-size:12px;color:#555;margin:0 0 6px;">'
    f'Інструкція: <b>{_prompt_esc}</b> &nbsp;|&nbsp;'
    f' steps={IP2P_STEPS} &nbsp;|&nbsp; text_gs={IP2P_TEXT_GS}'
    f' &nbsp;|&nbsp; img_gs={IP2P_IMG_GS} &nbsp;|&nbsp; {_ip2p_elapsed:.1f} сек'
    '</p>'
))

display(HTML(f'''
<div style="font-family:sans-serif;max-width:700px;user-select:none;">
  <p style="color:#555;font-size:13px;margin:4px 0 6px;">
    &#8592; ESRGAN (вхід IP2P) &nbsp;/&nbsp; AFTER IP2P &#8594;
  </p>
  <div id="ip2p-cmp" style="position:relative;display:block;width:100%;max-width:700px;
       cursor:col-resize;border-radius:8px;overflow:hidden;box-shadow:0 2px 8px rgba(0,0,0,.3);">
    <img src="data:image/jpeg;base64,{_b64_aft}" style="display:block;width:100%;height:auto;">
    <div id="ip2p-clip" style="position:absolute;top:0;left:0;width:50%;height:100%;overflow:hidden;">
      <img src="data:image/jpeg;base64,{_b64_bef}" style="width:200%;max-width:none;height:auto;display:block;">
    </div>
    <div id="ip2p-line" style="position:absolute;top:0;left:50%;width:3px;height:100%;
         background:#fff;box-shadow:0 0 6px rgba(0,0,0,.5);">
      <div style="position:absolute;top:50%;left:50%;transform:translate(-50%,-50%);
           background:#fff;border-radius:50%;width:34px;height:34px;
           display:flex;align-items:center;justify-content:center;
           box-shadow:0 0 6px rgba(0,0,0,.4);font-size:18px;color:#333;">&#8660;</div>
    </div>
    <div style="position:absolute;bottom:8px;left:8px;background:rgba(74,144,226,.85);
         color:#fff;padding:3px 10px;border-radius:4px;font-size:12px;font-weight:bold;">ESRGAN</div>
    <div style="position:absolute;bottom:8px;right:8px;background:rgba(155,89,182,.85);
         color:#fff;padding:3px 10px;border-radius:4px;font-size:12px;font-weight:bold;">IP2P</div>
  </div>
  <input type="range" min="0" max="100" value="50" id="ip2p-slider"
         style="width:100%;max-width:700px;margin-top:8px;accent-color:#9b59b6;">
</div>
<script>
(function(){{
  var c=document.getElementById('ip2p-cmp');
  var clip=document.getElementById('ip2p-clip');
  var line=document.getElementById('ip2p-line');
  var sl=document.getElementById('ip2p-slider');
  function set(p){{p=Math.max(0,Math.min(100,p));clip.style.width=p+'%';line.style.left=p+'%';sl.value=p;}}
  sl.addEventListener('input',function(){{set(+this.value);}});
  var dragging=false;
  function pos(e){{var r=c.getBoundingClientRect();var x=(e.touches?e.touches[0].clientX:e.clientX)-r.left;set(x/r.width*100);}}
  c.addEventListener('mousedown',function(e){{dragging=true;pos(e);}});
  document.addEventListener('mouseup',function(){{dragging=false;}});
  document.addEventListener('mousemove',function(e){{if(dragging)pos(e);}});
  c.addEventListener('touchstart',function(e){{pos(e);}},{{passive:true}});
  c.addEventListener('touchmove',function(e){{pos(e);e.preventDefault();}},{{passive:false}});
}})();
</script>'''))

# ── Скачування ───────────────────────────────────────────────────
with open(_ip2p_out, 'rb') as _fh:
    _ip2p_dl_b64 = base64.b64encode(_fh.read()).decode()

display(HTML(
    '<div style="background:#f5eeff;color:#2d1a4a;border-radius:8px;padding:10px;margin-top:10px;'
    'border-left:4px solid #9b59b6;font-family:sans-serif;">'
    '<b>📱 На телефоні:</b> кнопка нижче → браузер збере файл.'
    '</div>'
    + f'<a href="data:image/jpeg;base64,{_ip2p_dl_b64}" download="ip2p_result.jpg" '
    f'style="display:inline-block;margin:6px 4px;padding:11px 18px;'
    f'background:#9b59b6;color:#fff;font-weight:bold;font-size:14px;'
    f'border-radius:8px;text-decoration:none;">⬇️ IP2P результат</a>'
))

try:
    from google.colab import files as _colab_files
    _colab_files.download(_ip2p_out)
except Exception:
    pass

print('\n✅ IP2P готово!')
